# Проверка качества данных: Справочник факторов проблемности

**Файл:** `ref_book_fp.csv`  
**Содержание:** справочник ФП/СФП — расшифровка номеров, классификация по типу и признаку дефолта

## Загрузка данных

In [ ]:
import pandas as pd
import numpy as np

FILE_PATH = "/home/jovyan/documents/DMUKP_FP_DEF/ref_book_fp.csv"

KEY_COLUMNS = ["№", "Наименование", "ФП", "СФП", "Дефолт"]

FIELD_DESCRIPTIONS = {
    "№": "Номер фактора проблемности в справочнике (соответствует NUMBER_FP_SFP в CRM, ref_book_fp_id в H2.0)",
    "Наименование": "Текстовое описание фактора проблемности",
    "ФП": "Флаг: является ли данный номер фактором проблемности (Y/N)",
    "СФП": "Флаг: является ли данный номер существенным фактором проблемности (Y/N)",
    "Дефолт": "Флаг: является ли данный фактор показателем дефолта (Y/N)",
}

In [ ]:
df = pd.read_csv(FILE_PATH, dtype=str)
df.columns = df.columns.str.strip()

print(f"Строк: {len(df):,}")
print(f"Колонок: {len(df.columns)}")
print(f"\nВсе колонки в файле: {list(df.columns)}")
df.head()

## 1. Список ключевых полей с описанием

In [ ]:
fields_data = []
for i, col in enumerate(KEY_COLUMNS, start=1):
    present = "✓" if col in df.columns else "✗"
    desc = FIELD_DESCRIPTIONS.get(col, "(описание отсутствует)")
    fields_data.append({"№": i, "Поле": col, "Наличие": present, "Описание": desc})

df_fields = pd.DataFrame(fields_data)
df_fields.style.hide(axis="index")

## 2. Пропуски и пустые значения (ключевые колонки)

In [ ]:
total = len(df)

nulls_data = []
for col in KEY_COLUMNS:
    if col not in df.columns:
        continue
    s = df[col]
    cnt_nan = s.isna().sum()
    s_str = s.dropna().astype(str).str.strip().str.lower()
    cnt_none = (s_str == "none").sum()
    cnt_empty = (s_str == "").sum()
    cnt_total = cnt_nan + cnt_none + cnt_empty
    pct = cnt_total / total * 100

    nulls_data.append({
        "Поле": col,
        "NaN": cnt_nan,
        "None": cnt_none,
        "Пустые": cnt_empty,
        "Итого": cnt_total,
        "%": round(pct, 1),
    })

df_nulls = pd.DataFrame(nulls_data)
df_nulls.style.hide(axis="index")

## 3. Уникальность номера ФП

In [ ]:
num_col = df["№"].dropna().astype(str).str.strip()
num_col = num_col[num_col != ""]

n_total = len(num_col)
n_unique = num_col.nunique()
n_dupes = n_total - n_unique

print(f"Всего записей с номером ФП: {n_total}")
print(f"Уникальных номеров:         {n_unique}")
print(f"Дубликатов:                 {n_dupes}")

if n_dupes > 0:
    counts = num_col.value_counts()
    dup_detail = counts[counts > 1].reset_index()
    dup_detail.columns = ["№", "Кол-во"]
    print("\nПовторяющиеся номера:")
    display(dup_detail)

## 4. Проверка допустимых значений флагов

In [ ]:
for flag_col in ["ФП", "СФП", "Дефолт"]:
    if flag_col not in df.columns:
        print(f"{flag_col}: колонка отсутствует")
        continue
    vals = df[flag_col].dropna().astype(str).str.strip().value_counts()
    print(f"--- {flag_col} ---")
    print(vals.to_string())
    unexpected = set(vals.index) - {"Y", "N"}
    if unexpected:
        print(f"  ⚠ Недопустимые значения: {unexpected}")
    print()

## 5. Статистика по типам

In [ ]:
fp_y = (df["ФП"].str.strip() == "Y").sum() if "ФП" in df.columns else 0
sfp_y = (df["СФП"].str.strip() == "Y").sum() if "СФП" in df.columns else 0
def_y = (df["Дефолт"].str.strip() == "Y").sum() if "Дефолт" in df.columns else 0
both_fp_sfp = 0
if "ФП" in df.columns and "СФП" in df.columns:
    both_fp_sfp = ((df["ФП"].str.strip() == "Y") & (df["СФП"].str.strip() == "Y")).sum()

print(f"Всего записей в справочнике: {len(df)}")
print(f"ФП = Y:                      {fp_y}")
print(f"СФП = Y:                     {sfp_y}")
print(f"Одновременно ФП=Y и СФП=Y:   {both_fp_sfp}")
print(f"Дефолт = Y:                  {def_y}")

## 6. ФП/СФП с признаком дефолта

In [ ]:
mask_default = df["Дефолт"].astype(str).str.strip() == "Y"
df_default = df.loc[mask_default, KEY_COLUMNS].copy().reset_index(drop=True)

print(f"Всего ФП/СФП с Дефолт=Y: {len(df_default)}")
display(df_default)